# Agentic AI pipeline

## 1. Libraries

In [1]:
import json
import re
import logging
import time

logging.basicConfig(level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger("SmartAgent")

### Insight

With Logging one can get every step the agent took to get the asnwer which action it chose what input it received whether it retried etc.

We can Debug wrong outputs by seeing where the agent did wrong action.

## 2. Calculator Function

In [2]:
def calculator(expression: str) -> str:
    try:
        cleaned = re.sub(r'[^0-9+\-*/().\s]', '', expression)
        cleaned = cleaned.strip()
        if not cleaned:
            return "Error: No valid math expression found"
        result = eval(cleaned)
        return str(result)
    except ZeroDivisionError:
        return "Error: Division by zero"
    except Exception as e:
        return f"Error in calculation: {e}"

### Insight:
Calculator function take the input as string then evaluates math expressions.
if the expression is is not a correct extession it will throw "No valid math expression found" it returns the string output too.


## 3. Keyword Extractor


In [3]:
def extract_keywords(text: str) -> list:
    try:
        stopwords = {
            "the", "is", "are", "was", "were", "and", "or", "but", "for",
            "from", "with", "about", "this", "that", "these", "those",
            "what", "which", "who", "whom", "extract", "keywords", "please"
        }
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        keywords = [w for w in words if len(w) > 3 and w not in stopwords]
        seen = set()
        unique_keywords = []
        for k in keywords:
            if k not in seen:
                seen.add(k)
                unique_keywords.append(k)
        return unique_keywords[:5]
    except Exception as e:
        logger.error(f"Keyword extraction failed: {e}")
        return []

### Smarter Extraction
This function takes the string input and returns the list of unique words. It extracts keywords from text by removing stopwords and short words. If the word is less than 4 it will not be selected.

## 4 dditional Function

In [4]:
def word_counter(text: str) -> dict:
    try:
        words = text.split()
        sentences = re.split(r'[.!?]+', text)
        sentences = [s for s in sentences if s.strip()]
        return {
            "words": len(words), "characters": len(text), "sentences": len(sentences)
        }
    except Exception as e:
        return {"error": str(e)}


def reverse_text(text: str) -> str:
    try:
        return text[::-1]
    except Exception as e:
        return f"Error: {e}"


def unit_converter(query: str) -> str:
    try:
        query = query.lower()
        number_match = re.search(r'[-+]?\d*\.?\d+', query)
        if not number_match:
            return "Error: No number found in query"
        num = float(number_match.group())

        if "km" in query and "mile" in query:
            return f"{num} km = {round(num * 0.621371, 2)} miles"
        elif "kg" in query and ("pound" in query or "lb" in query):
            return f"{num} kg = {round(num * 2.20462, 2)} pounds"
        elif "celsius" in query and "fahrenheit" in query:
            return f"{num}C = {round((num * 9/5) + 32, 2)}F"
        else:
            return "Error: Unsupported unit conversion. Try km-miles, kg-pounds, or celsius-fahrenheit."
    except Exception as e:
        return f"Error: {e}"

### Insight

All the above function takes a string as an input.
- word_counter() return the returns a dictonary as that contains length of words, characters, sentences.
- reverse_text() return just string that contain whole text in reverse format.
- unit_converter() reutns string that contain the original Number converted into asked format.

## 5. Agent

In [5]:
agent_stats = {
    "total_queries": 0,
    "successful": 0,
    "failed": 0,
    "tool_usage": {}
}


def detect_intent(query: str) -> str:
    q = query.lower()
    if any(word in q for word in ["calculate", "compute", "solve", "what is ", "+", "-", "*", "/"]):
        if re.search(r'\d', q):
            return "calculation"
    if "keyword" in q or "extract" in q:
        return "keywords"
    if "word count" in q or "count words" in q or "how many words" in q:
        return "word_count"
    if "reverse" in q:
        return "reverse"
    if "convert" in q or ("km" in q and "mile" in q) or ("kg" in q and "pound" in q) or ("celsius" in q and "fahrenheit" in q):
        return "convert"
    return "general"


def agent(query: str, max_retries: int = 2) -> dict:
    agent_stats["total_queries"] += 1
    logger.info(f"Received query: {query!r}")

    intent = detect_intent(query)
    logger.info(f"Detected intent: {intent}")
    agent_stats["tool_usage"][intent] = agent_stats["tool_usage"].get(intent, 0) + 1

    attempt = 0
    last_error = None
    while attempt <= max_retries:
        try:
            if intent == "calculation":
                result = calculator(query)
                if result.startswith("Error"):
                    raise ValueError(result)
                response = {"type": "calculation", "result": result}

            elif intent == "keywords":
                result = extract_keywords(query)
                response = {"type": "keywords", "result": result}

            elif intent == "word_count":
                result = word_counter(query)
                response = {"type": "word_count", "result": result}

            elif intent == "reverse":
                result = reverse_text(query)
                response = {"type": "reverse", "result": result}

            elif intent == "convert":
                result = unit_converter(query)
                if result.startswith("Error"):
                    raise ValueError(result)
                response = {"type": "conversion", "result": result}

            else:
                response = {
                    "type": "general",
                    "result": f"I received your query: '{query}'. I don't have a specific tool for this, but I'm here to help with math, keywords, word counts, reversing text, and unit conversions."
                }

            agent_stats["successful"] += 1
            logger.info(f"Success on attempt {attempt + 1}")
            return response

        except Exception as e:
            last_error = str(e)
            attempt += 1
            logger.warning(f"Attempt {attempt} failed: {e}. Retrying...")
            time.sleep(0.1)

    agent_stats["failed"] += 1
    logger.error(f"All {max_retries + 1} attempts failed.")
    return {
        "type": "error",
        "result": f"Failed after {max_retries + 1} attempts. Last error: {last_error}"
    }

### Insight

The agent:
1. Analyzes the query using detect_intent function.
2. based on the of intent it call the correct function.
3. Retries on failure will loop till max_tries.
4. The agent function Returns a structured JSON response.

## 6. Test Cases

In [6]:
queries = [
    "Calculate 20 + 5",
    "Calculate 100 / 4",
    "Extract keywords from Artificial Intelligence is transforming industries worldwide",
    "What is machine learning?",
    "How many words in this sentence about agents and tools?",
    "Reverse hello world",
    "Convert 10 km to miles",
    "Convert 25 celsius to fahrenheit",
    "Calculate 10 / 0",   # triggers retry loop
]

for q in queries:
    print(f"\nQuery: {q}")
    response = agent(q)
    print("Response:", json.dumps(response, indent=2))
    print("-" * 60)


Query: Calculate 20 + 5
Response: {
  "type": "calculation",
  "result": "25"
}
------------------------------------------------------------

Query: Calculate 100 / 4
Response: {
  "type": "calculation",
  "result": "25.0"
}
------------------------------------------------------------

Query: Extract keywords from Artificial Intelligence is transforming industries worldwide
Response: {
  "type": "keywords",
  "result": [
    "artificial",
    "intelligence",
    "transforming",
    "industries",
    "worldwide"
  ]
}
------------------------------------------------------------

Query: What is machine learning?
Response: {
  "type": "general",
  "result": "I received your query: 'What is machine learning?'. I don't have a specific tool for this, but I'm here to help with math, keywords, word counts, reversing text, and unit conversions."
}
------------------------------------------------------------

Query: How many words in this sentence about agents and tools?
Response: {
  "type": "

ERROR:SmartAgent:All 3 attempts failed.


Response: {
  "type": "error",
  "result": "Failed after 3 attempts. Last error: Error: Division by zero"
}
------------------------------------------------------------


### Insight:
I've run some batch of test queries to make sure each route works.

We can see the log lines between the print statements:
- Types states the intent taken to solve the query.
- The Calculate 10 / 0 shows retry warnings. This proves that cycle is working.

## 7. Evaluation Metrics

In [7]:
def print_metrics():
    total = agent_stats["total_queries"]
    if total == 0:
        print("No queries processed yet.")
        return
    completion_rate = (agent_stats["successful"] / total) * 100
    print("=" * 50)
    print("AGENT PERFORMANCE METRICS")
    print("=" * 50)
    print(f"Total queries       : {total}")
    print(f"Successful          : {agent_stats['successful']}")
    print(f"Failed              : {agent_stats['failed']}")
    print(f"Task completion rate: {completion_rate:.2f}%")
    print("\nTool usage (cost proxy):")
    for tool, count in sorted(agent_stats["tool_usage"].items(), key=lambda x: -x[1]):
        print(f"  {tool:15s} : {count} call(s)")
    print("=" * 50)

print_metrics()

AGENT PERFORMANCE METRICS
Total queries       : 9
Successful          : 8
Failed              : 1
Task completion rate: 88.89%

Tool usage (cost proxy):
  calculation     : 3 call(s)
  convert         : 2 call(s)
  keywords        : 1 call(s)
  general         : 1 call(s)
  word_count      : 1 call(s)
  reverse         : 1 call(s)


## 8. Agent Interation

In [9]:
print("Smart Agent is ready! Type 'exit' to quit or 'metrics' to see stats.\n")

while True:
    try:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() == "exit":
            print("Goodbye!")
            break
        if user_input.lower() == "metrics":
            print_metrics()
            continue
        response = agent(user_input)
        print("Agent:", json.dumps(response, indent=2))
        print("-" * 40)
    except KeyboardInterrupt:
        print("\nInterrupted. Exiting...")
        break

Smart Agent is ready! Type 'exit' to quit or 'metrics' to see stats.

You: calculate 450/10
Agent: {
  "type": "calculation",
  "result": "45.0"
}
----------------------------------------
You: Convert 20km in m


ERROR:SmartAgent:All 3 attempts failed.


Agent: {
  "type": "error",
  "result": "Failed after 3 attempts. Last error: Error: Unsupported unit conversion. Try km-miles, kg-pounds, or celsius-fahrenheit."
}
----------------------------------------
You: convert 20km to miles
Agent: {
  "type": "conversion",
  "result": "20.0 km = 12.43 miles"
}
----------------------------------------
You: exit
Goodbye!
